<a href="https://colab.research.google.com/github/francesco-vaselli/NPTwins2026_Hackathon/blob/main/short/02_conditional_flow_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧲 Notebook 2 — Conditional Flow Matching on CMS Jet Data (Short Version)

**Duration:** ~1.75 hours &nbsp;·&nbsp; **Prerequisites:** Notebook 1 (Flow Matching fundamentals)

In Notebook 1 you built Flow Matching from scratch on 2D toy data. Now we flip a single switch — **conditioning** — and suddenly the same machinery becomes a surrogate for expensive detector simulation at the LHC. 🎯

> **About this version.** Short-form variant: we provide the sinusoidal time embedding instead of asking you to write it (it's a small standalone helper). Conversely, the **`ConditionalVectorField`** task gets extra scaffolding so you can focus on the architecture. At the end you build a **Wasserstein-based scorecard** — exactly the schema used by our submission validator — and we explain the optional leaderboard PR if you want to keep iterating after the session.

### 🎬 The physics problem, in one paragraph

At the CMS experiment, simulating the detector response for a single event costs minutes of CPU time per event. A neural **conditional generator** that learns

$$p(\mathbf{x}_{\text{reco}} \mid \mathbf{x}_{\text{gen}})$$

— the distribution of *reconstructed* jet features given *generator-level* jet features — can replace that Monte Carlo with a forward pass of a neural network. Orders of magnitude faster.

### 🎯 What you'll walk away with

1. A concrete **data pipeline** for CMS jet pairs (gen, reco), with preprocessing handled for you 🧼
2. A **`ConditionalVectorField`** network — MLP + residual blocks + conditioning — built from scratch with extra guidance 🧠
3. A **conditional CFM loss** + training loop that produces a real, working fast simulator in under 5 minutes 🏋️
4. A sanity-check **conditioning vs unconditional** comparison 🎇
5. A **rigorous scorecard** (Wasserstein distances + classifier two-sample test + b-tag AUC delta) — the same numbers our leaderboard validator expects 📏
6. An **optional path forward**: if you finish early, take the scorecard and try to beat it (more epochs, bigger model, Heun sampler, …) and submit a PR to the leaderboard 🏆

### 🧪 How this notebook is structured

- Each section introduces a concept, then asks you to implement a building block (`Task X.Y`).
- Right after each task, a cell calls a function from `test_conditional_flow.py` (or `test_evaluation.py` for the scorecard) — run it to validate before moving on! ✅
- The solution notebook `02_conditional_flow_model_solution.ipynb` is in the same folder. 💪

### 📚 Symbols

| Symbol | Meaning |
|---|---|
| 🎯 | Goal of the section |
| ✏️ | Task to implement |
| 💡 | Hint |
| ⚠️ | Common pitfall |
| 🧪 | Validation cell |
| 🚀 | Let's run it! |
| 🎁 | Provided for you (just run) |
| 🏆 | Leaderboard / optional |


## ⚙️ Setup

> **Running on Google Colab?** Run the install cell below — it's a no-op on machines where the packages are already installed, and it also downloads the dataset and the test suite.
> **Running locally?** Activate your tutorial env (see `README.md`), then download `data.npy` into this folder (see the command in the next cell).

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch', 'numpy', 'scipy', 'scikit-learn',
                           'matplotlib', 'torchdiffeq'])
    import urllib.request
    for fname in ('test_conditional_flow.py', 'test_evaluation.py', 'utils.py'):
        if not os.path.exists(fname):
            urllib.request.urlretrieve(
                f'https://raw.githubusercontent.com/fvaselli/NP_Twins/main/{fname}',
                fname,
            )
    print('✅ Colab environment ready.')
else:
    print('✅ Local environment — skipping install.')

# Download the dataset if not present (≈ 150 MB)
if not os.path.exists('data.npy'):
    print('📥 Downloading CMS jet dataset from Zenodo (~150 MB, one-off)...')
    import urllib.request
    urllib.request.urlretrieve(
        'https://zenodo.org/records/11126625/files/gen_ttbar_400k_final.npy',
        'data.npy',
    )
    print('✅ Dataset downloaded.')
else:
    print('✅ data.npy already present.')


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


We import the NB2 test suite (model + loss + training tests) and one helper from the evaluation suite (scorecard test) — run the `test_*` calls that follow each task.

In [ ]:
from test_conditional_flow import (
    test_sinusoidal_embedding,
    test_conditional_vector_field_model,
    test_conditional_fm_loss,
    test_train_conditional_model,
    test_generate_reco,
    run_all_tests_nb2,
)
from test_evaluation import test_compute_scorecard


## 1. The CMS Jet Dataset 🥽

The dataset is a NumPy array of 400k top-pair events, each row representing **one jet** with both its **generator-level** truth features (what nature produced) and its **reconstruction-level** measured features (what the detector saw).

**Reco features** (what the detector measures):
| Col | Name | Meaning |
|---|---|---|
| 0 | `btag` | b-tagging discriminator score (continuous, roughly ∈ [0, 1]) |
| 1 | `pt` | transverse momentum [GeV] |
| 2 | `phi` | azimuthal angle [rad] |
| 3 | `eta` | pseudorapidity |
| 4 | `N_const` | number of constituents (integer) |
| 5 | `ctag` | charm-tagging discriminator |

**Gen features** (the "conditions" that drive the generation):
| Col | Name | Meaning |
|---|---|---|
| 0 | `pt` | transverse momentum [GeV] |
| 1 | `eta` | pseudorapidity |
| 2 | `phi` | azimuthal angle [rad] |
| 3 | `E` | energy [GeV] |
| 4 | `flavour` | 0 (light), 1 (charm), 2 (bottom) |
| 5 | `muonsInJet` | number of muons inside the jet |

**We only tackle 3 reco features in this notebook** — `btag`, `pt`, `N_const` — so that training fits in well under 5 minutes on CPU. 🥖


### 🎁 Provided: data loading

Nothing for you to implement here — we give you a `DataExtractor` that slices the right columns and relabels flavour. Just run the cell.

In [ ]:
class DataExtractor:
    # Load and extract gen/reco features from the jet dataset.

    def __init__(self, data_path, n_samples=None):
        self.data = np.load(data_path, allow_pickle=True)
        if n_samples is not None:
            self.data = self.data[:n_samples]

        # Reco: btag, pt, phi, eta, N_const, ctag
        self.reco_features = ['btag', 'pt', 'phi', 'eta', 'N_const', 'ctag']
        self.reco = self.data[:, [5, 6, 7, 8, 10, 19]]

        # Gen: pt, eta, phi, E, flavour, muonsInJet
        self.gen_features = ['pt', 'eta', 'phi', 'E', 'flavour', 'muonsInJet']
        self.gen = self.data[:, [0, 1, 2, 3, 4, 9]]

        # Collapse flavour: 0=light, 1=charm, 2=bottom
        flav = np.abs(self.gen[:, 4])
        flav = np.where(np.isin(flav, [1, 2, 3, 21]), 0, flav)  # light + gluons
        flav = np.where(flav == 4, 1, flav)   # charm
        flav = np.where(flav == 5, 2, flav)   # bottom
        self.gen[:, 4] = flav

    def get_reco(self): return self.reco
    def get_gen(self):  return self.gen


extractor = DataExtractor('data.npy', n_samples=400000)
reco_data = extractor.get_reco()
gen_data  = extractor.get_gen()

print(f'Reco data shape: {reco_data.shape}  — features: {extractor.reco_features}')
print(f'Gen data shape:  {gen_data.shape}  — features: {extractor.gen_features}')
print(f'\nFlavour mix: light={np.mean(gen_data[:,4]==0):.1%}, '
      f'charm={np.mean(gen_data[:,4]==1):.1%}, '
      f'bottom={np.mean(gen_data[:,4]==2):.1%}')


### 🎁 Provided: a peek at the data

Just look at the plots — no coding needed. Notice the **long-tailed `pt` distribution** (challenging for a generator) and the **correlation between reco and gen `pt`** (the single most important thing the model has to learn).

In [ ]:
from utils import plot_1dhistos

plot_1dhistos(reco_data, extractor.reco_features)
plot_1dhistos(gen_data, extractor.gen_features)

# Scatter: reco_pt vs gen_pt — the main correlation to learn
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(gen_data[:5000, 0], reco_data[:5000, 1], s=1, alpha=0.3, c='steelblue')
ax.plot([0, 500], [0, 500], 'r--', lw=1, label='Perfect response')
ax.set_xlabel('gen $p_T$ [GeV]', fontsize=13)
ax.set_ylabel('reco $p_T$ [GeV]', fontsize=13)
ax.set_title('Detector response: reco $p_T$ vs gen $p_T$', fontsize=13)
ax.legend(); plt.tight_layout(); plt.show()


## 2. Preprocessing — Provided 🧼

Raw physics features have *wildly* different scales: `pt` ranges from tens to hundreds of GeV, `btag` lives in [0, 1], `N_const` is a small integer. Neural networks hate this. We apply a handful of standard tricks — standardising continuous features, **taking the ratio `reco_pt / gen_pt`** (so the model predicts a *detector correction* rather than absolute momentum), and **dequantising** `N_const` with uniform noise in [−0.5, 0.5] (so a continuous density model can generate a discrete count).

> **Why not ask you to implement this?** Because preprocessing is the least interesting part of the pipeline, and getting it subtly wrong silently breaks everything downstream. We'd rather spend your time on the model. 🎁


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


class Preprocessor:
    # Preprocess the 3-variable reco target [btag, pt_ratio, N_const].
    # - btag:    standard scale
    # - pt:      compute ratio reco_pt / gen_pt, then standard scale
    # - N_const: dequantise with U[-0.5, 0.5], then standard scale
    # - gen:     standard-scale continuous features (pt, eta, phi, E);
    #            leave flavour + muonsInJet as-is (discrete condition flags).

    def __init__(self):
        self.reco_scaler = StandardScaler()
        self.gen_scaler = StandardScaler()

    def transform(self, reco, gen, fit=False):
        wreco, wgen = np.copy(reco), np.copy(gen)

        btag     = wreco[:, 0]
        pt_ratio = wreco[:, 1] / wgen[:, 0]
        n_const  = wreco[:, 4] + np.random.uniform(-0.5, 0.5, len(wreco))

        reco_3 = np.stack([btag, pt_ratio, n_const], axis=1)

        if fit:
            reco_3 = self.reco_scaler.fit_transform(reco_3)
            wgen[:, :4] = self.gen_scaler.fit_transform(wgen[:, :4])
        else:
            reco_3 = self.reco_scaler.transform(reco_3)
            wgen[:, :4] = self.gen_scaler.transform(wgen[:, :4])
        return reco_3, wgen

    def inverse_transform(self, reco_t, gen_t):
        wreco, wgen = np.copy(reco_t), np.copy(gen_t)
        wreco = self.reco_scaler.inverse_transform(wreco)
        wgen[:, :4] = self.gen_scaler.inverse_transform(wgen[:, :4])

        # pt_ratio → pt  (multiply by gen_pt)
        wreco[:, 1] = wreco[:, 1] * wgen[:, 0]
        # N_const: dequantised → integer count
        wreco[:, 2] = np.rint(wreco[:, 2])
        return wreco, wgen


preprocessor = Preprocessor()

# Split first, then fit the scalers only on training data
reco_tr_raw, reco_vl_raw, gen_tr_raw, gen_vl_raw = train_test_split(
    reco_data, gen_data, test_size=0.2, random_state=42
)
reco_train, gen_train = preprocessor.transform(reco_tr_raw, gen_tr_raw, fit=True)
reco_val,   gen_val   = preprocessor.transform(reco_vl_raw, gen_vl_raw, fit=False)

print(f'Training set:   reco {reco_train.shape}, gen {gen_train.shape}')
print(f'Validation set: reco {reco_val.shape}, gen {gen_val.shape}')

# Move to device as tensors
reco_train_t = torch.tensor(reco_train, dtype=torch.float32).to(device)
gen_train_t  = torch.tensor(gen_train,  dtype=torch.float32).to(device)
reco_val_t   = torch.tensor(reco_val,   dtype=torch.float32).to(device)
gen_val_t    = torch.tensor(gen_val,    dtype=torch.float32).to(device)


## 3. Time Embedding — Provided ⏱️

In NB1 we fed the time `t` into the network as a single raw scalar. That works, but it's a *low-resolution* signal: the network has to squeeze a dense continuous function of `t` through a 1-dim input.

A standard upgrade (borrowed from Transformers) is the **sinusoidal embedding** — map the scalar `t` to a high-dimensional vector using a bank of sines and cosines at geometrically-spaced frequencies:

$$\text{embed}(t)_{2i}   = \sin(t \cdot \omega_i), \qquad \text{embed}(t)_{2i+1} = \cos(t \cdot \omega_i)$$

with $\omega_i = \exp\!\Big(-\frac{i \cdot \ln(10000)}{d/2}\Big)$.

> 🎁 **We provide this helper** — it's a 4-line standalone function. Just run the cell, then we'll plug it into the network in §4.


In [ ]:
def sinusoidal_embedding(t, dim):
    # Sinusoidal time embedding (provided).
    # Input:  t shape (batch, 1), dim (must be even)
    # Output: shape (batch, dim)
    half_dim = dim // 2
    freq = torch.exp(-torch.arange(half_dim, device=t.device, dtype=torch.float32)
                      * (np.log(10000.0) / half_dim))
    args = t * freq                              # (B, half_dim)
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


# Quick visualisation of a few channels
t_grid = torch.linspace(0, 1, 200).unsqueeze(1)
emb    = sinusoidal_embedding(t_grid, dim=16).numpy()

fig, ax = plt.subplots(figsize=(10, 4))
for d in range(8):
    ax.plot(t_grid.squeeze().numpy(), emb[:, d], label=f'dim {d}')
ax.set_xlabel('t'); ax.set_ylabel('embed(t)')
ax.set_title('First 8 channels of the sinusoidal time embedding')
ax.legend(ncol=4, fontsize=9, loc='lower right')
plt.tight_layout(); plt.show()


## 4. The Conditional Vector Field 🧠

This is the heart of the notebook. We need a network that takes

- $x_t$ — the current state along the flow, shape `(B, reco_dim)`
- $t$ — the time, shape `(B, 1)`
- $c$ — the gen-level condition, shape `(B, cond_dim)`

and outputs a velocity of shape `(B, reco_dim)`.

### 🧱 Architecture (the one we'll use)

```
   x_t ─┐
   t   ─┼─► sinusoidal_embedding  ─► concat ─► input_proj  ─► Linear+SiLU  ─┐
   c   ─┘                                                                    │
                                                                             │
                          ┌──────  residual blocks (× n_blocks)  ──────┐
                          │                                             │
                          ▼                                             ▼
                    h  ──► Linear+SiLU ──► Linear+SiLU ──► +h  ────► …
                          │                              ▲
                          └──────────────────────────────┘
                                                                             │
                                                                             ▼
                                                                       output_proj
                                                                       (→ reco_dim)
```

- **`input_proj`** maps `(x_t, time_embed, cond)` concatenated (total dim = `reco_dim + time_dim + cond_dim`) into the `hidden_dim`.
- Each **residual block** is a tiny MLP that is *added* to its input: `h = h + block(h)`. This keeps gradients flowing through deeper networks.
- **`output_proj`** is a single linear layer mapping back to `reco_dim` — no activation on the final layer (we want the velocity to take any real value).

### ✏️ Task 4.1 — Implement `ConditionalVectorField`

Build **both** `__init__` *and* `forward`. The scaffolding below walks you through both methods step-by-step — turn each numbered comment into one line of code.

**Signature**
```python
ConditionalVectorField(reco_dim=3, cond_dim=6, time_dim=16, hidden_dim=128, n_blocks=3)
```

**⚠️ Pitfalls**
- Use `nn.ModuleList` (*not* a plain Python list!) so PyTorch registers the residual-block parameters.
- The residual *addition* (`h = h + block(h)`) is what makes a block "residual" — don't forget the `h +`.
- If you forget to concat `cond`, the model ignores conditioning and becomes useless (the test catches this).
- If you forget to concat `t_emb`, the loss will plateau far above zero (also caught by the test).


In [ ]:
class ConditionalVectorField(nn.Module):
    # Conditional vector field with time embedding and residual blocks.

    def __init__(self, reco_dim=3, cond_dim=6, time_dim=16, hidden_dim=128, n_blocks=3):
        super().__init__()
        # Step 1) Save time_dim — forward() needs it to call sinusoidal_embedding.
        #   self.time_dim = time_dim
        #
        # Step 2) The total INPUT dim is x_t + time_embed + cond:
        #   input_dim = reco_dim + time_dim + cond_dim
        #
        # Step 3) input_proj: Linear(input_dim -> hidden_dim) followed by SiLU.
        #   self.input_proj = nn.Sequential(
        #       nn.Linear(input_dim, hidden_dim),
        #       nn.SiLU(),
        #   )
        #
        # Step 4) Residual blocks. Each is a small MLP:
        #         Linear(hidden -> hidden) -> SiLU -> Linear(hidden -> hidden) -> SiLU
        #         Wrap them in nn.ModuleList (NOT a plain list — PyTorch needs to see them).
        #   self.blocks = nn.ModuleList([
        #       nn.Sequential(
        #           nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
        #           nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
        #       )
        #       for _ in range(n_blocks)
        #   ])
        #
        # Step 5) output_proj: a single Linear(hidden_dim -> reco_dim). NO activation —
        #         velocities are unbounded reals.
        #   self.output_proj = nn.Linear(hidden_dim, reco_dim)
        #
        # YOUR CODE HERE — turn each step above into real code.
        raise NotImplementedError("Build the network in __init__! (Task 4.1)")

    def forward(self, x_t, t, cond):
        # Step 1) Embed t into a higher-dim time vector.
        #   t_emb = sinusoidal_embedding(t, self.time_dim)
        #
        # Step 2) Concatenate the three signals along the last dim:
        #   h = torch.cat([x_t, t_emb, cond], dim=-1)
        #
        # Step 3) Project to hidden width.
        #   h = self.input_proj(h)
        #
        # Step 4) Apply each residual block (add the block output back into h):
        #   for block in self.blocks:
        #       h = h + block(h)
        #
        # Step 5) Project to reco_dim and return.
        #   return self.output_proj(h)
        #
        # YOUR CODE HERE
        raise NotImplementedError("Implement forward! (Task 4.1)")


#### 🧪 Validate your model

In [ ]:
test_conditional_vector_field_model(ConditionalVectorField);


🚀 Sanity check: build one and count its parameters.

In [ ]:
reco_dim = 3
cond_dim = 6

model = ConditionalVectorField(
    reco_dim=reco_dim, cond_dim=cond_dim,
    time_dim=16, hidden_dim=128, n_blocks=3,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')

x_test = torch.randn(4, reco_dim, device=device)
t_test = torch.rand(4, 1, device=device)
c_test = torch.randn(4, cond_dim, device=device)
out = model(x_test, t_test, c_test)
print(f'Input:  x={tuple(x_test.shape)}, t={tuple(t_test.shape)}, cond={tuple(c_test.shape)}')
print(f'Output: {tuple(out.shape)}')


## 5. The Conditional CFM Loss 📉

Exactly the same recipe as NB1, with one tiny change — the model now also receives the condition:

$$\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t,\, z \sim p_{\text{data}}(\text{reco}, \text{gen}),\, x_0 \sim \mathcal{N}(0, I)}\Big[\big\lVert u_\theta(x_t, t,\, \text{gen}) - (x_1 - x_0) \big\rVert^2\Big]$$

with $x_1 = \text{reco}$, $x_t = (1 - t)\, x_0 + t\, x_1$, and $t \sim \text{Unif}[0, 1]$.

### ✏️ Task 5.1 — Implement `conditional_fm_loss(model, reco_batch, gen_batch)`

**Specification**
- **Inputs:** `model` (takes `x, t, cond`), `reco_batch` of shape `(B, reco_dim)`, `gen_batch` of shape `(B, cond_dim)`
- **Returns:** scalar MSE loss

**💡 Hints**
- Same code as NB1's `flow_matching_loss`, but pass `gen_batch` as the third arg to `model`.
- `torch.randn_like(reco_batch)` gives noise with the right shape and device.
- `t` is per-sample: shape `(batch, 1)`.


In [ ]:
def conditional_fm_loss(model, reco_batch, gen_batch):
    # Conditional Flow Matching loss.
    #
    # Args:
    #   model:      ConditionalVectorField  (takes x, t, cond)
    #   reco_batch: Target reco, shape (B, reco_dim)
    #   gen_batch:  Conditioning gen,  shape (B, cond_dim)
    # Returns:
    #   scalar MSE loss

    # YOUR CODE HERE
    # 1. x0 = torch.randn_like(reco_batch)
    # 2. t  ~ Uniform[0, 1]  with shape (batch, 1) on the right device
    # 3. x_t = (1 - t) * x0 + t * reco_batch
    # 4. target = reco_batch - x0
    # 5. predicted = model(x_t, t, gen_batch)
    # 6. return mean((predicted - target) ** 2)
    raise NotImplementedError("Implement the conditional CFM loss! (Task 5.1)")


In [ ]:
test_conditional_fm_loss(conditional_fm_loss, model_cls=ConditionalVectorField);


## 6. Training Loop 🏋️

Nothing exotic — shuffle, mini-batch, gradient step. Same as any PyTorch training loop.

### ✏️ Task 6.1 — Implement `train_conditional_model`

**Specification**
```python
train_conditional_model(model, reco_train, gen_train, reco_val, gen_val,
                        n_epochs=80, batch_size=512, lr=1e-3)
    -> (train_losses, val_losses)
```
- For each of `n_epochs` epochs:
  1. Shuffle the training indices with `torch.randperm`
  2. Iterate over mini-batches
  3. For each batch: compute `conditional_fm_loss`, backprop, optimizer step
  4. At the end of the epoch, compute a **val loss** (on a subsample of val is fine — 5000 points is plenty)
- Print progress every 10 epochs
- Return `(train_losses, val_losses)` — one entry per epoch each

**💡 Hints**
- Use `torch.optim.Adam(model.parameters(), lr=lr)`
- `for i in range(0, n_train, batch_size): idx = indices[i:i+batch_size]`
- Wrap the val loss in `torch.no_grad()` and put the model in `eval()` mode for that step


In [ ]:
def train_conditional_model(model, reco_train, gen_train, reco_val, gen_val,
                            n_epochs=80, batch_size=512, lr=1e-3):
    # Train the conditional flow matching model.
    #
    # Returns:
    #   train_losses: list[float], one avg training loss per epoch
    #   val_losses:   list[float], one val loss per epoch

    # YOUR CODE HERE
    raise NotImplementedError("Implement the training loop! (Task 6.1)")


In [ ]:
test_train_conditional_model(train_conditional_model, model_cls=ConditionalVectorField);


### 🚀 Train on CMS jets

We train for 80 epochs with batch size 512 — that's about **4 minutes on CPU**, a few seconds on GPU. ☕

In [ ]:
print('Starting training...')
train_losses, val_losses = train_conditional_model(
    model,
    reco_train_t, gen_train_t,
    reco_val_t,   gen_val_t,
    n_epochs=80, batch_size=512, lr=1e-3,
)
print('Training complete!')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label='Train', color='steelblue', alpha=0.8)
ax.plot(val_losses,   label='Val',   color='tomato',    alpha=0.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Conditional Flow Matching — training curves')
ax.legend(); plt.tight_layout(); plt.show()


## 7. Conditional Generation 🎬

The flow-matching ODE is exactly the same as NB1, but now the learned vector field takes the condition `cond` as an extra input at every step:

$$\frac{d x}{d t} = u_\theta(x, t, \text{gen})$$

To avoid OOM when generating for the full 80k-jet validation set, we process the data in chunks.

### ✏️ Task 7.1 — Implement `generate_reco`

**Specification**
```python
@torch.no_grad()
generate_reco(model, gen_cond, n_steps=100, batch_size=4096) -> (N, reco_dim) tensor
```
- Integrate the ODE from `t=0` to `t=1` using Euler steps of size `h = 1/n_steps`.
- Process `gen_cond` in chunks of at most `batch_size` rows.
- At each step: `x = x + h * model(x, t_tensor, cond_batch)`
- The starting point is pure noise: `torch.randn(chunk_size, reco_dim, device=gen_cond.device)`.

**💡 Hints**
- Infer `reco_dim` from `model.output_proj.out_features` — saves an extra argument.
- Use `torch.full((chunk_size, 1), step * h, device=...)` for the time tensor.


In [ ]:
@torch.no_grad()
def generate_reco(model, gen_cond, n_steps=100, batch_size=4096):
    # Generate reco features conditioned on gen features.
    #
    # Returns: tensor of shape (N, reco_dim)

    # YOUR CODE HERE
    raise NotImplementedError("Implement conditional generation! (Task 7.1)")


In [ ]:
test_generate_reco(generate_reco, model_cls=ConditionalVectorField);


### 🚀 Generate reco samples for the full val set, then compare

In [ ]:
print('Generating samples for the validation set...')
generated_scaled = generate_reco(model, gen_val_t, n_steps=100).cpu().numpy()
print(f'Generated: {generated_scaled.shape}')

# Inverse-transform both generated and target back to physical space
generated_phys, gen_val_phys = preprocessor.inverse_transform(generated_scaled, gen_val)
target_phys,    _            = preprocessor.inverse_transform(reco_val,         gen_val)

feature_names = ['b-tagging', r'$p_T$ [GeV]', r'$N_{\mathrm{const}}$']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (ax, name) in enumerate(zip(axes, feature_names)):
    ax.hist(target_phys[:, i],    bins=60, histtype='step', lw=2, color='steelblue', label='Target',    density=True)
    ax.hist(generated_phys[:, i], bins=60, histtype='step', lw=2, color='tomato',    label='Generated', density=True)
    ax.set_title(name, fontsize=14); ax.legend(fontsize=10)
    if i > 0:
        ax.set_yscale('log')
plt.suptitle('Generated vs target — 3 reco variables in physical space', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()


Not bad! The gross features of each distribution are recovered. Let's now check the model on a *conditional* diagnostic — that's where the real win shows up. ⤵️

## 8. 🎇 Does Conditioning Actually Help?

Quick experiment: train an **unconditional** baseline that ignores `gen` and learns only the **marginal** $p(\text{reco})$, then compare it to our conditional model on the b-tagging score, **split by jet flavour**.

### 🎁 Provided: training a baseline (no coding — just run)

We train a tiny unconditional baseline for 30 epochs (~1 min) by passing **zeros** as the condition. No need to re-implement the network.

In [ ]:
print('Training an unconditional baseline (cond fed as zeros)...')
baseline = ConditionalVectorField(
    reco_dim=reco_dim, cond_dim=cond_dim,
    time_dim=16, hidden_dim=128, n_blocks=3,
).to(device)

zero_train = torch.zeros_like(gen_train_t)
zero_val   = torch.zeros_like(gen_val_t)

_ = train_conditional_model(
    baseline, reco_train_t, zero_train,
    reco_val_t, zero_val,
    n_epochs=30, batch_size=512, lr=1e-3,
)


### 🏷️ The cleaner test: **b-tagging score, split by jet flavour**

A clean probe is the **b-tagging discriminator, conditioned on true jet flavour**. Physically:

- **bottom** jets (flavour = 2) should peak at **high** btag scores
- **light** jets (flavour = 0) should peak at **low** btag scores
- **charm** jets (flavour = 1) should sit somewhere in between

The conditional model sees `flavour` in its `cond` vector, so it *can* produce different btag distributions per flavour. The unconditional model sees nothing — it must generate the **same marginal btag distribution** regardless of true flavour.

In [ ]:
print('Generating from both models...')
gen_conditional   = generate_reco(model,    gen_val_t, n_steps=100).cpu().numpy()
gen_unconditional = generate_reco(baseline, zero_val,  n_steps=100).cpu().numpy()

cond_phys, _    = preprocessor.inverse_transform(gen_conditional,   gen_val)
uncond_phys, _  = preprocessor.inverse_transform(gen_unconditional, gen_val)

flavour = gen_val_phys[:, 4].astype(int)

flav_labels = {0: 'Light jets (u/d/s/g)', 1: 'Charm jets', 2: 'Bottom jets'}
bins = np.linspace(0, 1, 41)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, flav in zip(axes, [0, 1, 2]):
    mask = (flavour == flav)
    ax.hist(target_phys[mask, 0], bins=bins, histtype='step', lw=2,
            color='black',     label='Target',             density=True)
    ax.hist(cond_phys[mask, 0],   bins=bins, histtype='step', lw=2,
            color='tomato',    label='Conditional',        density=True)
    ax.hist(uncond_phys[mask, 0], bins=bins, histtype='step', lw=2,
            color='steelblue', label='Unconditional',      density=True, linestyle='--')
    ax.set_title(f'{flav_labels[flav]}  (n={mask.sum():,})', fontsize=12)
    ax.set_xlabel('b-tagging score'); ax.set_yscale('log')
    ax.legend(fontsize=9)
plt.suptitle('b-tag score split by true jet flavour — the unconditional model cannot tell them apart', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()


### 📝 What to look for

- The **target** (black) shows a clear flavour-dependent structure: light jets cluster at low btag, bottom jets at high btag, charm in between.
- The **conditional model** (tomato) tracks the black curve in *each* panel — it has learned $p(\text{btag} \mid \text{flavour}, \ldots)$ per flavour class.
- The **unconditional model** (blue dashed) is the **same curve in all three panels** — it only knows the flavour-averaged btag marginal, so it mis-tags everything.

That mismatch is exactly the kind of failure mode that would ruin a downstream b-tagging analysis. Conditioning is what makes the simulator usable for physics. 🎯

## 9. The Scorecard 📏

Eyeballing histograms is fine for a sanity check, but for *iteration* we need **numbers**. Three complementary metrics give a multi-angle picture of how well the simulator matches the data:

| Metric | Reads like… | Good when it is… |
|---|---|---|
| **Wasserstein distance per feature** `ws_per_feature` + sum | "Average ground distance" between the generated and target 1-D histograms — units: same as the feature. | Small (≈ 0). |
| **Classifier Two-Sample Test** `c2st` = \|AUC − 0.5\| | Trains a BDT to tell model samples from real samples. 0 ⇒ indistinguishable. | Small (≈ 0). |
| **b-tag AUC delta** `auc_delta_btag` | Difference between the b-vs-light AUC computed on **target** btag scores and on **generated** btag scores. | Small (≈ 0). |

The **`ws_sum`** is our **single leaderboard number** (lower = better).

### 🌊 What is the Wasserstein-1 distance?

For two 1-D distributions $P$ and $Q$, the **Wasserstein-1 (Earth Mover) distance** is

$$W_1(P, Q) = \int_{-\infty}^{\infty} \big| F_P(x) - F_Q(x) \big| \, dx$$

where $F_P, F_Q$ are the CDFs. Intuitively, it's the minimum amount of "earth" you'd have to move to turn one histogram into the other. Three things make it the right choice here:

1. It has the **same units** as the feature (a btag-score Wasserstein of `0.05` and a $p_T$ Wasserstein of `5 GeV` mean comparable things in their own currencies).
2. Unlike KL or χ², it doesn't blow up when the two distributions have disjoint support.
3. `scipy.stats.wasserstein_distance(a, b)` computes it from raw samples in one line — no histogramming, no binning choice.

We compute `W_1` per feature and then sum across features for a single number.

### 📋 The schema (matches our submission validator)

The scorecard is a `dict` with exactly these keys — the same schema enforced by `scripts/validate_submission.py`:

```python
{
    "ws_per_feature": [w_btag, w_pt, w_nconst],   # list[float], length = reco_dim
    "ws_sum":         float,                       # = sum(ws_per_feature)
    "c2st":           float,                       # = |BDT AUC − 0.5|, in [0, 0.5]
    "auc_delta_btag": float,                       # = |AUC_target − AUC_model|
}
```

### ✏️ Task 9.1 — Implement `compute_scorecard`

**Specification**
```python
compute_scorecard(generated, target, gen,
                  c2st_n=20000, random_state=42)
    -> dict with keys ws_per_feature, ws_sum, c2st, auc_delta_btag
```

**💡 Hints**
- `scipy.stats.wasserstein_distance(a, b)` computes 1-D WS between two 1-D arrays.
- For C2ST, concatenate `target[:n]` (label 1) and `generated[:n]` (label 0), then fit a `HistGradientBoostingClassifier(max_iter=100, random_state=random_state)` on a 70/30 split. Use `roc_auc_score` on the held-out 30%. `c2st = |AUC − 0.5|`.
- For the b-tag AUC delta, mask to jets with **flavour ∈ {0, 2}** (light + bottom), label b-jets as 1, and use the **btag score (column 0)** as the classifier output. Compute the ROC AUC twice — once with `target[:, 0]` as the score, once with `generated[:, 0]`. Return `|AUC_target − AUC_model|`.
- `c2st_n` caps the number of rows fed to the BDT — keeps repeated scoring fast.

**⚠️ Pitfalls**
- Don't pass the full 80 k validation set to the BDT; it'll be slow. `c2st_n = 20000` is plenty.
- `ws_sum` must equal `sum(ws_per_feature)` exactly — the test checks it.


In [ ]:
from scipy.stats import wasserstein_distance
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split


def compute_scorecard(generated, target, gen,
                      c2st_n=20000, random_state=42):
    # Build a dict of evaluation metrics for a (generated, target, gen) triple.
    #
    # Args
    # ----
    # generated : np.ndarray (N, reco_dim) — model samples in physical space
    # target    : np.ndarray (N, reco_dim) — real reco samples in physical space
    # gen       : np.ndarray (N, cond_dim) — gen-level conditions (flavour at col 4)
    # c2st_n    : int — max rows for the BDT (for speed)
    #
    # Returns
    # -------
    # dict with keys: ws_per_feature, ws_sum, c2st, auc_delta_btag

    # YOUR CODE HERE
    # 1. Compute per-feature Wasserstein distances and their sum.
    #    ws = [wasserstein_distance(target[:, i], generated[:, i]) for i in range(d)]
    # 2. Run the BDT-based C2ST on the first c2st_n rows of each.
    #    - X = concat(target[:n], generated[:n]); y = concat(ones(n), zeros(n))
    #    - 70/30 train/test split, HistGradientBoostingClassifier(max_iter=100), AUC on test
    #    - c2st = |AUC - 0.5|
    # 3. Compute the b-tag AUC delta on jets with flavour in {0, 2}
    #    using column 0 (btag) as the score.
    #    - mask = (flav == 2) | (flav == 0)
    #    - y_true = (flav[mask] == 2).astype(int)
    #    - auc_t = roc_auc_score(y_true, target[mask, 0])
    #    - auc_m = roc_auc_score(y_true, generated[mask, 0])
    #    - auc_delta_btag = |auc_t - auc_m|
    # 4. Return {ws_per_feature, ws_sum, c2st, auc_delta_btag}
    raise NotImplementedError("Build the scorecard! (Task 9.1)")


In [ ]:
test_compute_scorecard(compute_scorecard);


### 🚀 Score your trained model

This gives you the baseline numbers — these are what you'd be trying to beat if you take the optional leaderboard challenge below.

In [ ]:
scorecard = compute_scorecard(generated_phys, target_phys, gen_val_phys)

reco_3_names = ['btag', r'$p_T$ [GeV]', r'$N_{\mathrm{const}}$']

print('─' * 60)
print('Scorecard — your trained NB2 model')
print('─' * 60)
for i, v in enumerate(scorecard['ws_per_feature']):
    print(f'  WS [{reco_3_names[i]:<18}]: {v:.4f}')
print(f'  ws_sum                : {scorecard["ws_sum"]:.4f}')
print(f'  c2st                  : {scorecard["c2st"]:.4f}')
print(f'  auc_delta_btag        : {scorecard["auc_delta_btag"]:.4f}')
print('─' * 60)
print()
print('Lower is better on every line. ws_sum is the single leaderboard number.')


## 10. Final Comprehensive Test ✅

Run the NB2 test suite one more time. We pass the **provided** `sinusoidal_embedding` so the orchestrator can validate it as well — even though it wasn't a student task this time.

In [ ]:
run_all_tests_nb2(
    sinusoidal_embedding=sinusoidal_embedding,
    model_cls=ConditionalVectorField,
    conditional_fm_loss=conditional_fm_loss,
    train_conditional_model=train_conditional_model,
    generate_reco=generate_reco,
);


## 11. 🏆 Optional — The Leaderboard Challenge

> **Strictly optional, no time pressure.** If you've made it here and the session is winding down, you can stop — you've built a working conditional Flow Matching detector simulator and scored it rigorously, which was the goal. The next part is purely for participants who want to keep iterating after the session and prove how much they can improve on the baseline.

The repo at `https://github.com/fvaselli/NPTwins2026_Hackathon` has a tiny public **leaderboard** ranking submissions by `ws_sum` (lower = better). If you want in:

### How a submission works

A submission is **two files** sharing one `<id>`:

```
submissions/scorecards/<your-id>.json     ← the numbers (what gets ranked)
submissions/notebooks/<your-id>.ipynb     ← your executed notebook (what gets audited)
```

The `<id>` is your choice — typically `<github-handle>__<short-run-tag>`, e.g. `alice__deep-512-200ep`.

### Step-by-step

1. **Fork** the repo on GitHub.
2. **Run a longer / better experiment** (see ideas below). Keep this notebook running.
3. **Re-run** the scorecard cell on your new generated samples.
4. **Save the JSON** with the cell below — fill in your name and run tag.
5. **Save this notebook** *with cells executed* (outputs visible) as `submissions/notebooks/<your-id>.ipynb`. **Don't overwrite** the upstream `short/02_conditional_flow_model.ipynb` — that one stays as the canonical baseline.
6. **Open a PR** that adds the two files. A small CI check (`scripts/validate_submission.py`) verifies the schema; a maintainer eyeballs the notebook and merges.
7. On merge, a GitHub Action regenerates `LEADERBOARD.md`, sorted by `ws_sum`.

### Save your submission JSON


In [ ]:
import json, time

# ⬇️ EDIT ME if you're submitting
your_name = 'your-github-handle'
run_tag   = 'baseline-80ep'
notes     = 'NB2 short baseline — 80 epochs, hidden_dim=128, n_blocks=3, Euler 100'

submission = {
    'id':        f'{your_name}__{run_tag}',
    'name':      your_name,
    'run_tag':   run_tag,
    'notes':     notes,
    'timestamp': int(time.time()),
    'config': {
        'path':      'linear',
        'sampler':   'euler',
        'n_steps':   100,
        'arch':      {'reco_dim': reco_dim, 'cond_dim': cond_dim,
                      'time_dim': 16, 'hidden_dim': 128, 'n_blocks': 3},
        'n_epochs':  80,
    },
    'scorecard': scorecard,
}

os.makedirs('submissions/scorecards', exist_ok=True)
os.makedirs('submissions/notebooks',  exist_ok=True)
out_path = f'submissions/scorecards/{submission["id"]}.json'
with open(out_path, 'w') as f:
    json.dump(submission, f, indent=2)

print(f'✅ Wrote {out_path}')
print(f'   ws_sum         : {scorecard["ws_sum"]:.4f}   (lower = better)')
print(f'   c2st           : {scorecard["c2st"]:.4f}')
print(f'   auc_delta_btag : {scorecard["auc_delta_btag"]:.4f}')
print()
print(f'📓 To submit: also save THIS notebook (with cells executed) as')
print(f'        submissions/notebooks/{submission["id"]}.ipynb')
print(f'   then open a PR adding both files.')


### 💡 Ideas for a better submission

All of these are legal — anything is fair as long as the scorecard is computed honestly on the val split:

- **More epochs** — the loss is still trending down at epoch 80; try 150–200.
- **Bigger model** — `hidden_dim=256`, `n_blocks=5` typically helps the tails.
- **Heun sampler** — implement the conditional Heun integrator (you wrote the unconditional version in NB1, the conditional one only adds `cond` to each `model(...)` call). Often a free win at low NFE.
- **More ODE steps** — `n_steps=200` instead of 100; cheap, sometimes helps.
- **Different probability path** — try the **trigonometric** path: replace the linear interpolation with $x_t = \cos(\pi t/2)\, x_0 + \sin(\pi t/2)\, x_1$ and target $v_t = \tfrac{\pi}{2}(\cos(\pi t/2)\, x_1 - \sin(\pi t/2)\, x_0)$. Variance-preserving — sometimes noticeably better on tails.
- **Lower learning rate + longer run** — `lr=3e-4` for 200+ epochs can clean up tails.
- **Better preprocessing** — the `pt_ratio` is one choice; you could try `log(reco_pt)` as the target instead.

> **Cheating** is anything that makes `ws_sum` go down without actually improving the simulator: scoring `target` against itself, scoring against training data, etc. The executed notebook in your PR is the audit trail. ⚠️

Have fun, and may the lowest `ws_sum` win. 🥇

## 🏁 Summary

In this notebook you:

1. Loaded and explored the **CMS jet dataset** (400k jet pairs, 3 reco vars + 6 gen vars).
2. Used a ready-made **Preprocessor** and a provided **sinusoidal time embedding**.
3. Built a **`ConditionalVectorField`** network from scratch — input projection, residual blocks, output projection — with time + condition wired into the input.
4. Wrote a **conditional CFM loss**, a **training loop**, and a **batched generation function**.
5. Trained a real fast detector simulator in well under 5 minutes of CPU time 🚀.
6. Compared against an **unconditional baseline** and saw concretely *why* the conditioning signal matters.
7. Built a rigorous **Wasserstein-based scorecard** matching the leaderboard schema — and (optionally) learned how to submit.

Thanks for sticking with it — and if you decide to push for the leaderboard later, godspeed. 🎯
